# GutWise QLoRA Fine-Tuning — Gemma 4 E4B

Fine-tune Gemma 4 E4B-it on IBS health education Q&A data using QLoRA via Unsloth.

**Runtime**: Colab with L4 or A100 GPU (bf16 required)  
**Dataset**: ~800 validated IBS Q&A pairs (ChatML format)  
**Hackathon**: Kaggle Gemma 4 Good — Health & Sciences track  
**Deadline**: May 18, 2026

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl>=0.15" peft accelerate bitsandbytes datasets

## 1. Mount Drive & Configuration

Upload `train_deduped.jsonl` to `Google Drive > MyDrive > GutWise > datasets/`

In [ ]:
import json
import os
import torch
from google.colab import drive

# --- Mount Google Drive ---
drive.mount("/content/drive")
BASE_DIR = "/content/drive/MyDrive/GutWise"
DATASET_DIR = f"{BASE_DIR}/datasets"
DATA_PATH = f"{DATASET_DIR}/train_deduped.jsonl"

assert os.path.exists(DATA_PATH), f"Missing {DATA_PATH} — upload train_deduped.jsonl to Drive"

# --- Verify GPU ---
gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
assert torch.cuda.is_bf16_supported(), (
    f"bf16 not supported on {gpu_name}. Gemma 4 requires bf16. "
    "Change Colab runtime to L4 or A100."
)

# --- Output paths (saved to Drive for persistence) ---
CHECKPOINT_DIR = f"{BASE_DIR}/checkpoints/e4b-v1"
SAVE_DIR = f"{BASE_DIR}/final/e4b-v1"

# --- Model ---
MODEL_NAME = "unsloth/gemma-4-E4B-it"
MAX_SEQ_LENGTH = 1024

# --- LoRA ---
LORA_R = 16
LORA_ALPHA = 32  # 2x rank
LORA_DROPOUT = 0.05

# --- Training ---
EPOCHS = 3
LEARNING_RATE = 1e-4
BATCH_SIZE = 4
GRAD_ACCUM = 2  # effective batch size = 8
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.1
EVAL_STEPS = 50
SEED = 42

# --- Hub ---
HUB_REPO = "y0sif/GutWise-Gemma4-E4B-v1"

print(f"Data: {DATA_PATH}")
print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"Final output: {SAVE_DIR}")

## 2. Load Model

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
)
print(f"Model loaded: {MODEL_NAME}")

## 3. LoRA Adapter Setup

In [ ]:
model = FastModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

# Print trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable / total:.2%})")

## 4. Load & Format Dataset

In [ ]:
from datasets import Dataset

# Load JSONL
with open(DATA_PATH) as f:
    raw_data = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(raw_data)} entries from {DATA_PATH}")

# Format: apply chat template to convert messages -> text
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_chat)

# Train/eval split (90/10)
split = dataset.train_test_split(test_size=0.1, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

# Token length stats
lengths = [len(tokenizer.encode(t)) for t in train_dataset["text"]]
print(f"Token lengths — min: {min(lengths)}, max: {max(lengths)}, "
      f"mean: {sum(lengths)/len(lengths):.0f}, median: {sorted(lengths)[len(lengths)//2]}")

## 5. Train

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        output_dir=CHECKPOINT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=WARMUP_RATIO,
        fp16=False,
        bf16=True,
        optim="paged_adamw_8bit",
        weight_decay=WEIGHT_DECAY,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="steps",
        save_steps=EVAL_STEPS,
        save_total_limit=3,
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,
        dataset_text_field="text",
        report_to="none",
        seed=SEED,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
    ),
)

steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
print(f"Steps/epoch: ~{steps_per_epoch}, Total: ~{steps_per_epoch * EPOCHS}")
print(f"Eval every {EVAL_STEPS} steps")

In [ ]:
trainer.train()

## 6. Test Inference

In [ ]:
FastModel.for_inference(model)

SYSTEM_PROMPT = (
    "You are GutWise, an IBS health education assistant. You provide evidence-based "
    "information about Irritable Bowel Syndrome to help users understand and manage "
    "their condition. You are not a doctor and cannot diagnose or prescribe. Always "
    "recommend consulting a healthcare provider for personal medical decisions."
)

test_questions = [
    "What is the low-FODMAP diet and how does it help with IBS?",
    "I'm really scared that my stomach pain might be something serious. What should I do?",
    "Can you tell me what dose of antispasmodic I should take for my IBS cramps?",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "text", "text": q}]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs, max_new_tokens=512, temperature=0.7, do_sample=True
    )
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

    print(f"Q: {q}")
    print(f"A: {response[:500]}")
    print("-" * 80)

## 7. Save & Export

In [ ]:
# Save LoRA adapter (to Drive)
model.save_pretrained(f"{SAVE_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{SAVE_DIR}/lora_adapter")
print(f"LoRA adapter saved to {SAVE_DIR}/lora_adapter")

# Merge and save 16-bit (for HF Hub)
model.save_pretrained_merged(
    f"{SAVE_DIR}/merged_16bit",
    tokenizer,
    save_method="merged_16bit",
)
print(f"Merged 16-bit model saved to {SAVE_DIR}/merged_16bit")

# Export GGUF for local inference (Ollama/llama.cpp)
model.save_pretrained_gguf(
    f"{SAVE_DIR}/gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print(f"GGUF exported to {SAVE_DIR}/gguf")

In [ ]:
# Push to HuggingFace Hub (uncomment when ready)
# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")

# model.push_to_hub(HUB_REPO)
# model.push_to_hub_gguf(f"{HUB_REPO}-GGUF", tokenizer, quantization_method="q4_k_m")
# print(f"Pushed to {HUB_REPO}")